### Import librerie


In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import xgboost as xgb
import keras
import json
from keras import layers
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
)

### Data Utilities


In [2]:
# Costanti
HYPO = 70.0
HYPER = 180.0
L_BOUND = 40.0
U_BOUND = 400.0

In [3]:
def load_splits(splits_dir="data/split_sets"):
    """Carica gli split e i metadati dai relativi file.
    Args:
        splits_dir (str): Directory contenente gli split set (default: 'data/split_sets')
    Returns:
        tuple: (train_set, val_set, test_set, X_cols, y_cols)
    """
    datasets = []
    for name in ["train", "val", "test"]:
        df = pd.read_parquet(f"{splits_dir}/{name}_set.parquet")
        datasets.append(df)

    with open(f"{splits_dir}/metadata.json", "r") as f:
        metadata = json.load(f)

    return tuple(datasets + [metadata["X_cols"], metadata["y_cols"]])

In [4]:
def rescale_data(df, rescale_cols):
    """Effettua il rescale dei dati back al loro intervallo originario.
    Args:
        df (pd.DataFrame): DataFrame con i dati da riscalare
        rescale_cols (list): List dei nomi delle colonne da riscalare
    Returns:
        pd.DataFrame: Dataframe con le colonne scelte riscalate
    """
    df = df.copy()
    for col in rescale_cols:
        df[col] = ((df[col] + 1) * (U_BOUND - L_BOUND) / 2) + L_BOUND
    return df

In [5]:
def calculate_metrics(df):
    """Calcola le metriche per ciascun paziente in un sottoinsieme.
    Args:
        df (pd.DataFrame): Dataframe con le colonne 'Patient_ID', 'target', e 'y_pred'
    Returns:
        tuple: (samples, maes, mapes, rmses) - numero di samples e lista delle metriche ottenute
    """
    samples = 0
    maes, mapes, rmses = [], [], []

    for patient_id in df["Patient_ID"].unique():
        patient_data = df[df["Patient_ID"] == patient_id]
        if patient_data.empty:
            continue

        samples += len(patient_data)
        maes.append(mean_absolute_error(patient_data["target"], patient_data["y_pred"]))
        mapes.append(
            mean_absolute_percentage_error(
                patient_data["target"], patient_data["y_pred"]
            )
            * 100
        )
        rmses.append(
            root_mean_squared_error(patient_data["target"], patient_data["y_pred"])
        )

    return samples, maes, mapes, rmses

In [6]:
def print_results(df):
    """Stampa i risultati delle valutazioni cumulative e per condizione glicemica.
    Args:
        df (pd.DataFrame): DataFrame con valori inferiti e di riferimento
    """

    def print_metrics(title, samples, maes, mapes, rmses):
        """Stampa le metriche formattate per una specifica condizione."""
        if title != "Cumulative":
            print("~" * 10)
        print(title)
        print(f"Samples: {samples}")
        if maes:  # Stampa solo se abbiamo dei dati
            print(f"MAE: {np.mean(maes):.2f}({np.std(maes):.2f})")
            print(f"MAPE: {np.mean(mapes):.2f}({np.std(mapes):.2f})")
            print(f"RMSE: {np.mean(rmses):.2f}({np.std(rmses):.2f})")

    # Overall results
    samples, maes, mapes, rmses = calculate_metrics(df)
    print_metrics("Cumulative", samples, maes, mapes, rmses)

    # Results by condition
    for condition in ["Normal", "Hyper", "Hypo"]:
        condition_df = df[df["bgClass"] == condition]
        samples, maes, mapes, rmses = calculate_metrics(condition_df)
        print_metrics(condition, samples, maes, mapes, rmses)

### Dnn Utilities


In [7]:
def create_gru_model():
    """Costruisci il modello GRU"""
    return keras.Sequential(
        [
            layers.Input(shape=(8, 1)),
            layers.GRU(86, return_sequences=True),
            layers.Dropout(0.2),
            layers.GRU(86, return_sequences=False),
            layers.Dense(1),
        ],
        name="GRU_Model",
    )

In [8]:
def predict_in_batches(model, data, model_type, batch_size=256):
    """Esegui le predizione ed effettua il reshaping automatico dei dati"""
    if model_type not in ["mlp", "lstm", "gru"]:
        raise ValueError(f"Model type must be one of {['mlp', 'lstm', 'gru']}")

    def _reshape_for_rnn(X):
        """Esegui il reshape de i dati per i modelli RNN"""
        return X.reshape(X.shape[0], X.shape[1], 1)

    # Prepare data based on model type
    if model_type in ["lstm", "gru"]:
        data_reshaped = _reshape_for_rnn(data.values)
    else:
        data_reshaped = data.values

    return model.predict(data_reshaped, batch_size=batch_size, verbose=0)

### Training Dnn e Tml Utilities


In [9]:
def load_best_xgb_params(study_path="tuning/results/xgb_optuna_study.pkl"):
    """Carica i migliori iperparametri per XGB dallo studio di Optuna"""
    print("=" * 80)
    print("LOADING BEST XGBOOST HYPERPARAMETERS")
    print("=" * 80)

    if not os.path.exists(study_path):
        raise FileNotFoundError(f"XGBoost Optuna study not found at {study_path}")

    with open(study_path, "rb") as f:
        study = pickle.load(f)

    best_params = study.best_params.copy()
    best_mae = study.best_value

    print(f"Best validation MAE from tuning: {best_mae:.4f}")
    print(f"Best hyperparameters:")
    for param, value in best_params.items():
        if isinstance(value, float):
            print(f"  {param}: {value:.6f}")
        else:
            print(f"  {param}: {value}")

    # Aggiungi i parametri fissi per il training
    best_params.update({"random_state": 42, "device": "cuda:0"})

    return best_params, best_mae

In [10]:
def train_final_xgb_model(train_set, val_set, X_cols, y_cols, best_params):
    """Addestra il modelo XGB sul set formato da train+val con i migliori iperparametri passati"""
    print("\n" + "=" * 80)
    print("TRAINING FINAL XGBOOST MODEL")
    print("=" * 80)

    # Combine train and validation sets
    combined_set = pd.concat([train_set, val_set], ignore_index=True)
    print(f"Combined training set size: {len(combined_set)}")
    print(f"  - Original train set: {len(train_set)}")
    print(f"  - Original val set: {len(val_set)}")

    # Create and train model
    print(f"\nCreating XGBoost model with optimized hyperparameters...")
    model = xgb.XGBRegressor(**best_params)

    print(f"Training on combined train+val set...")
    X_combined = combined_set[X_cols]
    y_combined = combined_set[y_cols[-1]]

    model.fit(X_combined, y_combined)

    print(f"Training completed!")

    return model

In [11]:
def evaluate_xgb_on_test(model, test_set, X_cols, y_cols):
    """Valuta XGB sul test set."""
    print("\n" + "=" * 80)
    print("EVALUATING XGBOOST ON TEST SET")
    print("=" * 80)

    # Make predictions
    test_eval = test_set.copy()
    test_eval["y_pred"] = model.predict(test_eval[X_cols])

    # Prepare results
    test_eval = test_eval.rename(columns={y_cols[-1]: "target"})
    test_eval = rescale_data(test_eval, ["target", "y_pred"])

    # Select output columns
    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = test_eval[output_columns]

    # Print results
    print("XGBoost Test Set Results:")
    print_results(results)

    return results

In [12]:
def save_xgb_model_and_results(
    model, results, models_path="models/test_set", outputs_path="outputs/test_set"
):
    """Salva il modello XGB otteuto e i risultati sul test set."""
    print(f"\nSaving XGBoost model and results...")

    # Create directories if they don't exist
    os.makedirs(models_path, exist_ok=True)
    os.makedirs(outputs_path, exist_ok=True)

    # Save model
    model_path = f"{models_path}/xgb.pickle"
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"XGBoost model saved to: {model_path}")

    # Save results
    results_path = f"{outputs_path}/xgb_output.csv"
    results.to_csv(results_path, index=False)
    print(f"XGBoost test results saved to: {results_path}")

In [13]:
def load_gru_model_weights(weights_path="models/val_set/gru.weights.h5"):
    """Carica il modello GRU e i suoi pesi."""
    print("\n" + "=" * 80)
    print("LOADING GRU MODEL")
    print("=" * 80)

    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"GRU weights not found at {weights_path}")

    # Create GRU model with same architecture as training
    model = create_gru_model()

    # Load weights
    model.load_weights(weights_path)
    print(f"GRU model weights loaded from: {weights_path}")

    # Print model summary
    print(f"\nGRU Model Architecture:")
    model.summary()

    return model

In [14]:
def evaluate_gru_on_test(model, test_set, X_cols, y_cols):
    """Valuta GRU sul test set."""
    print("\n" + "=" * 80)
    print("EVALUATING GRU ON TEST SET")
    print("=" * 80)

    # Make predictions
    test_eval = test_set.copy()
    test_eval["y_pred"] = predict_in_batches(model, test_eval[X_cols], "gru").flatten()

    # Prepare results
    test_eval = test_eval.rename(columns={y_cols[-1]: "target"})
    test_eval = rescale_data(test_eval, ["target", "y_pred"])

    # Select output columns
    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = test_eval[output_columns]

    # Print results
    print("GRU Test Set Results:")
    print_results(results)

    return results

In [15]:
def save_gru_results(results, outputs_path="outputs/test_set"):
    """Salva i risultati di GRU sul test-set"""
    print(f"\nSaving GRU results...")

    # Create directory if it doesn't exist
    os.makedirs(outputs_path, exist_ok=True)

    # Save results
    results_path = f"{outputs_path}/gru_output.csv"
    results.to_csv(results_path, index=False)
    print(f"GRU test results saved to: {results_path}")

### Main pipeline


In [16]:
print("FINAL TEST SET EVALUATION")
print("=" * 80)
print("Evaluating XGBoost (with optimal hyperparameters) and GRU on test set")
print("=" * 80)

FINAL TEST SET EVALUATION
Evaluating XGBoost (with optimal hyperparameters) and GRU on test set


In [17]:
# Load data splits
print("Loading data splits...")
train_set, val_set, test_set, X_cols, y_cols = load_splits()

print(f"Dataset sizes:")
print(f"  Train set: {len(train_set)} samples")
print(f"  Val set: {len(val_set)} samples")
print(f"  Test set: {len(test_set)} samples")
print(f"  Features: {len(X_cols)}")
print(f"  Target: {y_cols}")

Loading data splits...
Dataset sizes:
  Train set: 66071 samples
  Val set: 9111 samples
  Test set: 18322 samples
  Features: 8
  Target: ['lead30']


In [18]:
# ===== VALUTAZONE XGBOOST =====

# Carica i migliori iperparametri ottenuti
best_params, best_val_mae = load_best_xgb_params()
# Addestra XGB
xgb_model = train_final_xgb_model(train_set, val_set, X_cols, y_cols, best_params)
# Valutalo sul test set
xgb_results = evaluate_xgb_on_test(xgb_model, test_set, X_cols, y_cols)
# Salva il modello e i risultati
save_xgb_model_and_results(xgb_model, xgb_results)

LOADING BEST XGBOOST HYPERPARAMETERS
Best validation MAE from tuning: 16.1101
Best hyperparameters:
  max_depth: 8
  min_child_weight: 0.109943
  subsample: 0.987964
  colsample_bytree: 0.932977
  reg_alpha: 0.000001
  reg_lambda: 0.000000
  learning_rate: 0.018660
  n_estimators: 374
  gamma: 0.000158

TRAINING FINAL XGBOOST MODEL
Combined training set size: 75182
  - Original train set: 66071
  - Original val set: 9111

Creating XGBoost model with optimized hyperparameters...
Training on combined train+val set...


c:\Users\cerch\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training completed!

EVALUATING XGBOOST ON TEST SET
XGBoost Test Set Results:
Cumulative
Samples: 18322
MAE: 15.81(4.13)
MAPE: 11.57(3.90)
RMSE: 21.23(5.61)


c:\Users\cerch\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:729: UserWarning: [12:57:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


~~~~~~~~~~
Normal
Samples: 10837
MAE: 14.81(4.66)
MAPE: 12.30(4.05)
RMSE: 19.70(6.09)
~~~~~~~~~~
Hyper
Samples: 6179
MAE: 19.48(6.57)
MAPE: 8.61(2.84)
RMSE: 25.38(8.14)
~~~~~~~~~~
Hypo
Samples: 1306
MAE: 17.24(7.34)
MAPE: 29.06(12.83)
RMSE: 19.24(7.76)

Saving XGBoost model and results...
XGBoost model saved to: models/test_set/xgb.pickle
XGBoost test results saved to: outputs/test_set/xgb_output.csv


In [19]:
# ===== VALUTAZIONE GRU =====

# Carica il modello GRU
gru_model = load_gru_model_weights()
# Valutalo sul test set
gru_results = evaluate_gru_on_test(gru_model, test_set, X_cols, y_cols)
# Salva i risultati
save_gru_results(gru_results)


LOADING GRU MODEL
GRU model weights loaded from: models/val_set/gru.weights.h5

GRU Model Architecture:


Model: "GRU_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 8, 86)          │        22,962 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8, 86)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 86)             │        44,892 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            87 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,941 (265.39 KB)

 Trainable params: 67,941 (265.39 KB)

 Non-trainable params: 0 (0.00 B)


EVALUATING GRU ON TEST SET
GRU Test Set Results:
Cumulative
Samples: 18322
MAE: 15.67(3.96)
MAPE: 11.53(3.84)
RMSE: 20.91(5.41)
~~~~~~~~~~
Normal
Samples: 10837
MAE: 14.93(4.91)
MAPE: 12.50(4.21)
RMSE: 19.78(6.46)
~~~~~~~~~~
Hyper
Samples: 6179
MAE: 19.28(6.12)
MAPE: 8.56(2.69)
RMSE: 25.06(7.49)
~~~~~~~~~~
Hypo
Samples: 1306
MAE: 15.16(6.21)
MAPE: 25.39(10.57)
RMSE: 17.22(6.93)

Saving GRU results...
GRU test results saved to: outputs/test_set/gru_output.csv


In [20]:
print("\n" + "=" * 80)
print("EVALUATION COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"Models saved in: models/test_set/")
print(f"Results saved in: outputs/test_set/")


EVALUATION COMPLETED SUCCESSFULLY!
Models saved in: models/test_set/
Results saved in: outputs/test_set/
